<a href="https://colab.research.google.com/github/Elicheralasravya/Assignments/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fetch Book Data from API

### Subtask:
Make an HTTP GET request to a public book API (e.g., Gutendex) to retrieve a list of books in JSON format. The JSON data should contain attributes like title, author, and publication year.


**Reasoning**:
To begin, I will import the `requests` library to enable HTTP requests, define the Gutendex API endpoint, make a GET request, and then convert the response into a JSON object. This consolidates the initial setup and data retrieval into a single, efficient step.



In [2]:
import requests

# Define the Gutendex API endpoint URL
api_url = "https://gutendex.com/books/"

# Make an HTTP GET request to the API endpoint
response = requests.get(api_url)

# Convert the HTTP response to JSON format
books_data = response.json()

# Print a snippet of the JSON data to verify
print(books_data['results'][0])

{'id': 84, 'title': 'Frankenstein; or, the modern prometheus', 'authors': [{'name': 'Shelley, Mary Wollstonecraft', 'birth_year': 1797, 'death_year': 1851}], 'summaries': ['"Frankenstein; Or, The Modern Prometheus" by Mary Wollstonecraft Shelley is a Gothic novel published in 1818. It tells the story of Victor Frankenstein, a young scientist who creates a living creature from assembled body parts in an unorthodox experiment. When the creature awakens, Victor flees in horror, abandoning his creation. The conscious being must navigate a world that fears him, learning language and seeking connection, only to face repeated rejection. Embittered and alone, the creature confronts his creator with a desperate request that will set both on a dark path of vengeance and tragedy. (This is an automatically generated summary.)'], 'editors': [], 'translators': [], 'subjects': ["Frankenstein's monster (Fictitious character) -- Fiction", 'Frankenstein, Victor (Fictitious character) -- Fiction', 'Gothi

## Initialize SQLite Database



**Reasoning**:
To initialize the SQLite database and create the 'books' table, I will write a Python code block that imports `sqlite3`, connects to `books.db`, creates a cursor, executes the `CREATE TABLE` command, commits the changes, and then closes the cursor and connection.



In [4]:
import sqlite3

# Connect to the SQLite database (creates it if it doesn't exist)
conn = sqlite3.connect('books.db')

# Create a cursor object
c = conn.cursor()

# Create the 'books' table if it doesn't already exist
c.execute('''
    CREATE TABLE IF NOT EXISTS books (
        title TEXT,
        author TEXT,
        publication_year INTEGER
    )
''')

# Commit the changes and close the connection
conn.commit()
c.close()
conn.close()

print("SQLite database 'books.db' initialized and 'books' table created successfully.")

SQLite database 'books.db' initialized and 'books' table created successfully.


## Store Book Data in Database

### Subtask:
Iterate through the parsed JSON data obtained from the API. For each book, extract its title, author, and publication year. Insert these details as a new record into the 'books' table in the SQLite database. Implement basic error handling and ensure data integrity.


**Reasoning**:
To store the extracted book data into the SQLite database, I will connect to the `books.db` database, iterate through the `books_data['results']` to extract the title, author, and set the publication year to `None`, and then insert each book's details into the `books` table using a parameterized SQL query to ensure data integrity.



In [6]:
import sqlite3

# Connect to the SQLite database
conn = sqlite3.connect('books.db')
c = conn.cursor()

# Iterate through the books data and insert into the database
for book in books_data['results']:
    title = book.get('title')

    # Extract author name, handle cases where authors list might be empty
    authors = book.get('authors', [])
    author = authors[0]['name'] if authors else None

    # Publication year is not directly available, setting to None as per instructions
    publication_year = None

    try:
        c.execute("INSERT INTO books (title, author, publication_year) VALUES (?, ?, ?)",
                  (title, author, publication_year))
    except sqlite3.Error as e:
        print(f"Error inserting book '{title}': {e}")

# Commit the changes and close the connection
conn.commit()
c.close()
conn.close()

print("Book data successfully inserted into 'books.db'.")

Book data successfully inserted into 'books.db'.


## Retrieve and Display Stored Data

### Subtask:
Execute a SQL query to select all records from the 'books' table in the SQLite database. Fetch the results and display them in a clear and readable format, such as a list of dictionaries or a pandas DataFrame.


**Reasoning**:
To retrieve and display the stored data, I will use `sqlite3` to connect to the database, execute a SELECT query, fetch the results, and then use `pandas` to display them in a DataFrame for clarity.



In [8]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect('books.db')
c = conn.cursor()

# Execute a SELECT query to retrieve all records from the 'books' table
c.execute("SELECT * FROM books")

# Fetch all the results
records = c.fetchall()

# Get column names from cursor description
column_names = [description[0] for description in c.description]

# Create a pandas DataFrame from the fetched records and column names
books_df = pd.DataFrame(records, columns=column_names)

# Print the DataFrame to display the stored book data
print("Books data stored in the database:")
print(books_df)

# Close the database connection
c.close()
conn.close()


Books data stored in the database:
                                                title  \
0             Frankenstein; or, the modern prometheus   
1                                   Wuthering Heights   
2                            Moby Dick; Or, The Whale   
3                                 Pride and Prejudice   
4                                    Romeo and Juliet   
..                                                ...   
59                            A Doll's House : a play   
60                                            Dracula   
61  The Importance of Being Earnest: A Trivial Com...   
62                     Adventures of Huckleberry Finn   
63                               A Tale of Two Cities   

                          author publication_year  
0   Shelley, Mary Wollstonecraft             None  
1                  Brontë, Emily             None  
2               Melville, Herman             None  
3                   Austen, Jane             None  
4           Shakespe